In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3

In [4]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [13]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
dataset = files.upload()

dataset_path = next(iter(dataset))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving merged_dataset.csv to merged_dataset (1).csv


In [7]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 04-18 07:03:20 [__init__.py:239] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.50.0.dev0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [8]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.language_model.model` require gradients


In [28]:
# ✅ Load Dataset

from datasets import load_dataset
dataset = load_dataset("csv", data_files=dataset_path)
train_test_split = dataset["train"].train_test_split(test_size=0.04, seed=42)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 5980
Test dataset size: 250


In [29]:
# ✅ Load Dataset
dataset = train_dataset

# ✅ Format dataset using Chat Template
def format_chat(example):
    return {
        "messages": [
            {"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"},
            {"role": "assistant", "content": f'Simplified Explanation: {example["simplified_text"]}'},
        ]
    }

dataset = dataset.map(format_chat)

In [30]:
dataset[1]['messages']



[{'content': "You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nPublic Law 114-194\n114th Congress\n\n                                 An Act\n\n\n \n To extend the termination of sanctions with respect to Venezuela under \n     the Venezuela Defense of Human Rights and Civil Society Act of \n               2014. <<NOTE: July 15, 2016 -  [S. 2845]>> \n\n    Be it enacted by the Senate and House of Representatives of the \nUnited States of America in Congress assembled, <<NOTE: Venezuela \nDefense of Human Rights and Civil Society Extension Act of 2016. 50 USC \n1701 note.>> \nSECTION 1. SHORT TITLE.\n\n    This Act may be cited as the ``Venezuela Defense of Human Rights and \nCivil S

In [31]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [32]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [33]:
dataset[100]

{'legal_text': 'SECTION 15 of Bombay Reorganisation Act, 1960\r\r\n15. Allocation of members.-\r\r\n(1) Every sitting member of the Legislative Assembly of Bombay representing a constituency which on the appointed day by virtue of the provisions of section 14 stands transferred, whether with or without alteration of boundaries, to the State of Gujarat shall, as from that day, cease to be a member of the Legislative Assembly of Bombay and shall be deemed to have been elected to the Legislative Assembly of Gujarat by that constituency as so transferred.\r\r\n(2) All other sitting members of the Legislative Assembly of Bombay shall become members of the Legislative Assembly of Maharashtra and any such sitting member representing a constituency the extent or the name and extent of which are altered by virtue of the provisions of section 14 shall be deemed to have been elected to the Legislative Assembly of Maharashtra by that constituency as so altered.\r\r\n(3) The sitting member of the L

We now have to apply the chat template for `Gemma-3` onto the conversations, and save it to `text`

In [34]:
def apply_chat_template(examples):
    texts = tokenizer.apply_chat_template(examples["messages"])
    return { "text" : texts }
pass
dataset = dataset.map(apply_chat_template, batched = True)

Map:   0%|          | 0/5980 [00:00<?, ? examples/s]

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [35]:
dataset[100]["text"]

'<bos><start_of_turn>user\nYou are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nSECTION 15 of Bombay Reorganisation Act, 1960\r\r\n15. Allocation of members.-\r\r\n(1) Every sitting member of the Legislative Assembly of Bombay representing a constituency which on the appointed day by virtue of the provisions of section 14 stands transferred, whether with or without alteration of boundaries, to the State of Gujarat shall, as from that day, cease to be a member of the Legislative Assembly of Bombay and shall be deemed to have been elected to the Legislative Assembly of Gujarat by that constituency as so transferred.\r\r\n(2) All other sitting members of the Legislative Assembly of Bombay 

In [36]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5980 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [37]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=2):   0%|          | 0/5980 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [38]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><bos><start_of_turn>user\nYou are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\nSECTION 15 of Bombay Reorganisation Act, 1960\r\r\n15. Allocation of members.-\r\r\n(1) Every sitting member of the Legislative Assembly of Bombay representing a constituency which on the appointed day by virtue of the provisions of section 14 stands transferred, whether with or without alteration of boundaries, to the State of Gujarat shall, as from that day, cease to be a member of the Legislative Assembly of Bombay and shall be deemed to have been elected to the Legislative Assembly of Gujarat by that constituency as so transferred.\r\r\n(2) All other sitting members of the Legislative Assembly of Bo

Now let's print the masked out example - you should see only the answer is present:

In [41]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                                                                                                                                                                                                                         Simplified Explanation: Simplified text:\r\nAllocation of Members:\r\n1. Members representing constituencies transferred to Gujarat will become members of the Legislative Assembly of Gujarat.\r\n2. Members representing other constituencies will become members of the Legislative Assembly of Maharashtra.\r\n3. The nominated member representing the Anglo-Indian community will continue to represent the community in the Legislative Assembly of Maharashtra.<end_of_turn>\n'

In [39]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.57 GB of memory reserved.


In [42]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,980 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248/4,000,000,000 (0.37% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.052000
2,2.934000
3,2.459400
4,3.148000
5,1.931500
6,1.588200
7,1.383600
8,1.345400
9,0.817200
10,1.133000


In [43]:
model.save_pretrained("/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier")
tokenizer.save_pretrained("/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier")

['/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier/processor_config.json']

In [78]:

import gc
tensor = None
gc.collect()
torch.cuda.empty_cache()
import gc
gc.collect()
torch.cuda.empty_cache()



In [77]:
print(torch.cuda.memory_summary(device=None, abbreviated=False))

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 2            |        cudaMalloc retries: 2         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   5544 MiB |  14376 MiB |  38405 GiB |  38400 GiB |
|       from large pool |   5413 MiB |  14096 MiB |  37700 GiB |  37695 GiB |
|       from small pool |    130 MiB |    279 MiB |    704 GiB |    704 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   5544 MiB |  14376 MiB |  38405 GiB |  38400 GiB |
|       from large pool |   5413 MiB |  14096 MiB |  37700 GiB |

In [46]:
!pip install unsloth evaluate bert_score rouge_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.3/124.3 kB 10.5 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=63c6ad7a3959f3e600c50d018e3dadb98cc157fc7da674bcf7bde4c34a065127
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.6
    Uninstalling protobuf-4.25.6:
      Successfully uninstalled protobuf-4.25.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 

In [92]:

from unsloth import FastLanguageModel
import evaluate
from transformers import pipeline


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/legal_nlp/gemma_legal_simplifier",  # Path to your fine-tuned model
    dtype = torch.float16,
    load_in_4bit = True,
    use_safetensors = True,
    device_map = "auto"
)

# ✅ Enable inference optimizations
FastLanguageModel.for_inference(model)









==((====))==  Unsloth 2025.3.19: Fast Gemma3 patching. Transformers: 4.50.0.dev0. vLLM: 0.8.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [97]:
# ✅ Load a small test subset
test_dataset = train_test_split["test"]

In [98]:
# ✅ Create generation prompts using chat template
def make_chat_prompt(legal_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": f"You are an expert in legal language and its interpretation. Your task is to simplify the following legal text while maintaining its original meaning and intent, so that a layman person can understand. The simplified version should be accessible to individuals without a legal background, using clear and concise language. Simplify the following legal text:\n{example['legal_text']}"}],
        tokenize=False,
        add_generation_prompt=True
    )

predictions, references = [], []

def simplify_text(legal_text):
    prompt = make_chat_prompt(legal_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8192,
            do_sample=False,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded

In [ ]:
for example in test_dataset:
    decoded = simplify_text(example["legal_text"])
    # Remove the prompt portion to get only the response
    simplified_output = decoded.split("\nmodel\n")[-1].strip()
    predictions.append(simplified_output.strip())
    references.append(example["simplified_text"].strip())


In [48]:

# ✅ Evaluate with ROUGE, BLEU, BERTScore
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")





🔍 ROUGE: {'rouge1': np.float64(0.12145437796839574), 'rouge2': np.float64(0.04116998725538082), 'rougeL': np.float64(0.0999918755330371), 'rougeLsum': np.float64(0.11634953647681667)}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 BERTScore: {'precision': [0.8608855605125427, 0.861850917339325, 0.8521679043769836, 0.7828741073608398, 0.8375384211540222, 0.8461451530456543, 0.7139588594436646, 0.8644603490829468, 0.861850917339325, 0.8521679043769836, 0.7828741073608398, 0.8375384211540222, 0.8461451530456543, 0.7139588594436646, 0.8644603490829468, 0.861850917339325, 0.8521679043769836, 0.7828741073608398, 0.8375384211540222, 0.8461451530456543], 'recall': [0.8336681127548218, 0.855929970741272, 0.8173779845237732, 0.7685176730155945, 0.8126885890960693, 0.8084638118743896, 0.7605316638946533, 0.833172082901001, 0.855929970741272, 0.8173779249191284, 0.7685177326202393, 0.8126885890960693, 0.8084636926651001, 0.7605316042900085, 0.8331720232963562, 0.855929970741272, 0.8173779249191284, 0.7685177326202393, 0.8126885890960693, 0.8084636926651001], 'f1': [0.8470582962036133, 0.8588802218437195, 0.8344104886054993, 0.7756294012069702, 0.8249263763427734, 0.8268753886222839, 0.7365097403526306, 0.8485279083251953,

In [96]:
rouge_score = rouge.compute(predictions=predictions, references=references)
bertscore_score = bertscore.compute(predictions=predictions, references=references, lang="en")
print("ROUGE Scores:", rouge_score)
print("BERTScore Scores:")
for metric, value in bertscore_score.items():
    try:
      print(f"{metric}: {sum(value)/len(value)}")
    except TypeError:
      print(f"{metric}: {value}")

import random
random_index = random.randint(0, len(predictions) - 1)
print(f"Random Index: {random_index}")
print(f"\n\n\nPrediction: {predictions[random_index]}")

print(f"\n\n\nReference: {references[random_index]}")




ROUGE Scores: {'rouge1': np.float64(0.27538756698348293), 'rouge2': np.float64(0.113358272932741), 'rougeL': np.float64(0.19408119610999625), 'rougeLsum': np.float64(0.23290956402849142)}
BERTScore Scores:
precision: 0.8041309416294098
recall: 0.8881392478942871
f1: 0.8439434468746185
hashcode: roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.50.0.dev0)
Random Index: 0



Prediction: Okay, here’s a simplified explanation of this legal text, geared towards someone who isn’t familiar with legal jargon:

**What This Law Does – Explained Simply**

This law officially names a post office building. It’s a straightforward piece of legislation designed to give a specific location a more formal name.

**Here’s how it works:**

* **The Location:** There’s a particular U.S. Postal Service (USPS) post office located at 55 South Pioneer Boulevard in Springboro, Ohio.
* **The New Name:**  This law changes the official name of that post office to the “Richard ‘Dick’ Chenault Post Office Building.”